# 05 Agent Loop / Harness From Scratch

**Goal:** see one tool round slowly, build a reusable bounded loop, then hand the orchestration to the OpenAI Agents SDK.

1. **See it:** inspect task → decision → action → observation → answer.
2. **Repeat it:** run two dependent tools, then observe a step limit and a recoverable error.
3. **Compare it:** use `Agent` + `Runner` on the original notes task and inspect its trace.

**New since Class 4:** Class 4 introduced tool calling. Here we own the repeated lifecycle, termination, failure observations, and trace.

> “Thought” means a short observable decision summary. We do not request or store private chain-of-thought.

提示：模型选择下一步；程序验证、执行、记录，并决定何时停止。


## References

- [OpenAI Agents SDK overview](https://developers.openai.com/api/docs/guides/agents)
- [OpenAI Agents SDK quickstart](https://developers.openai.com/api/docs/guides/agents/quickstart)
- [Running agents and the agent loop](https://developers.openai.com/api/docs/guides/agents/running-agents)
- [Run results and state](https://developers.openai.com/api/docs/guides/agents/results)
- [Agents SDK integrations and observability](https://developers.openai.com/api/docs/guides/agents/integrations-observability)
- [OpenAI function calling guide](https://developers.openai.com/api/docs/guides/function-calling)
- [OpenAI Traces dashboard](https://platform.openai.com/traces)
- [ReAct paper](https://arxiv.org/abs/2210.03629)

The official SDK describes `Runner` as a loop: call the model, execute tool calls, append tool results, and continue until final output or a turn limit. This notebook first implements that lifecycle directly with the Responses API.


## Setup and Run Order

```bash
uv sync --locked --extra dev
cp .env.example .env  # first setup only; keep your existing file
uv run --extra dev jupyter notebook
```

Fill in `OPENAI_API_KEY` in `.env`. These notebooks make real model calls; `LLM_PROVIDER=mock` only applies to the source-code demos/tests. Local failure experiments use an injected fake client and incur no API usage.

Select this repo's Python kernel. Use **Restart Kernel & Run All** for a clean run. To repeat the staged experiment, restart at “Task and Initial State”; re-running only the execution cell would append duplicate tool observations.

Keep synthetic classroom data only. SDK tracing may export prompts and tool data; set `OPENAI_AGENTS_DISABLE_TRACING=1` before starting the kernel if you want local-only traces.


In [ ]:
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

# Support launching Jupyter from the repo, notebooks/, or the course parent.
locations = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in locations if (p / "pyproject.toml").exists()
                     and (p / "src/harness/loop.py").exists()), None)
if PROJECT_ROOT is None:
    candidate = Path.cwd() / "repo-03-agent-harness"
    PROJECT_ROOT = candidate if (candidate / "pyproject.toml").exists() else None
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from repo-03-agent-harness or its notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
if not os.getenv("OPENAI_BASE_URL", "").strip():
    os.environ.pop("OPENAI_BASE_URL", None)
MODEL = os.getenv("LLM_MODEL", "gpt-5.4-mini")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)
print("OPENAI_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("MODEL:", MODEL)
print("openai:", version("openai"), "| openai-agents:", version("openai-agents"))
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Copy .env.example to .env, set OPENAI_API_KEY, then restart this cell.")
client = OpenAI(timeout=30, max_retries=1)


In [ ]:
# Both implementations use this same local knowledge source.
CLASS_NOTES = [
    {
        "title": "Agent Loop",
        "body": (
            "A ReAct-style agent loop alternates between model decisions, "
            "tool actions, observations, and a final answer."
        ),
    },
    {
        "title": "Harness",
        "body": (
            "The harness owns tool execution, state updates, step limits, "
            "error handling, and traces."
        ),
    },
]

class SearchNotesArgs(BaseModel):
    query: str = Field(min_length=1, description="Topic to search in the class notes.")

def search_class_notes(query: str) -> list[dict[str, str]]:
    """Search local notes about agent loops and harness responsibilities."""
    terms = set(query.lower().split())
    matches = [
        note
        for note in CLASS_NOTES
        if terms & set(f"{note['title']} {note['body']}".lower().split())
    ]
    return matches

search_schema = SearchNotesArgs.model_json_schema()
search_schema["additionalProperties"] = False
MANUAL_TOOLS = [{
    "type": "function",
    "strict": True,
    "name": "search_class_notes",
    "description": "Search local class notes about agent loops and harnesses.",
    "parameters": search_schema,
}]
LOCAL_TOOLS = {"search_class_notes": search_class_notes}

print(json.dumps(MANUAL_TOOLS[0], indent=2, ensure_ascii=False))


---

## Section 1 - Hand-Write the Loop and Watch the Trace Grow

We will run one query: **According to the local class notes, what is an agent loop and what does the harness own?**

The next **five code cells** are one continuous run. Each cell adds a new auditable stage to `manual_trace`.

```text
task -> model decision -> action -> observation -> final answer
```

This is the practical ReAct pattern. The loop does not need a literal free-form `Thought:` string. What matters is that the model chooses an action, the application executes it, and the observation returns to the model.


### Stage 1 of 5 - Task and Initial State

The harness creates conversation state and starts a trace before calling the model.


In [ ]:
QUERY = (
    "According to the local class notes, what is an agent loop "
    "and what does the harness own?"
)
MANUAL_INSTRUCTIONS = (
    "Answer from the local class notes. Use search_class_notes before answering. "
    "After receiving tool output, answer concisely."
)
conversation: list[Any] = [{"role": "user", "content": QUERY}]
manual_trace: list[dict[str, Any]] = [{"stage": "task", "input": QUERY}]

print(json.dumps(manual_trace, indent=2, ensure_ascii=False))


### Stage 2 of 5 - Model Decision

The first model call is forced to select a tool for a predictable classroom trace. At this point no Python tool has run. The trace records the observable decision, not hidden chain-of-thought.


In [ ]:
first_response = client.responses.create(
    model=MODEL,
    input=conversation,
    instructions=MANUAL_INSTRUCTIONS,
    tools=MANUAL_TOOLS,
    tool_choice="required",
    parallel_tool_calls=False,
    temperature=0.1,
    max_output_tokens=600,
)
if first_response.status != "completed":
    raise RuntimeError(f"First response was {first_response.status}; inspect output budget.")
function_calls = [item for item in first_response.output if item.type == "function_call"]
assert len(function_calls) == 1, "The staged demo requires one tool call."
manual_trace.append({
    "stage": "decision",
    "summary": "The model requested a tool instead of guessing.",
    "response_id": first_response.id,
    "output_item_types": [item.type for item in first_response.output],
})

print(json.dumps(manual_trace[-1], indent=2, ensure_ascii=False))


### Stage 3 of 5 - Validate the Action

A `function_call` is only a request. The harness must check the tool name, parse JSON, and validate arguments before execution.


In [ ]:
validated_calls: list[dict[str, Any]] = []

for call in function_calls:
    if call.name not in LOCAL_TOOLS:
        raise ValueError(f"Unknown tool requested: {call.name}")
    raw_arguments = json.loads(call.arguments or "{}")
    arguments = SearchNotesArgs.model_validate(raw_arguments).model_dump()
    validated_calls.append({
        "call_id": call.call_id,
        "name": call.name,
        "arguments": arguments,
    })
    manual_trace.append({
        "stage": "action", "tool": call.name, "arguments": arguments
    })

print(json.dumps(manual_trace[-1], indent=2, ensure_ascii=False))


### Stage 4 of 5 - Execute and Record the Observation

Python executes the approved function. The result becomes both a trace observation and a `function_call_output` item for the next model turn.


In [ ]:
conversation.extend(first_response.output)

for call in validated_calls:
    result = LOCAL_TOOLS[call["name"]](**call["arguments"])
    manual_trace.append({
        "stage": "observation", "tool": call["name"], "result": result
    })
    conversation.append({
        "type": "function_call_output",
        "call_id": call["call_id"],
        "output": json.dumps(result, ensure_ascii=False),
    })

print(json.dumps(manual_trace[-1], indent=2, ensure_ascii=False))


### Stage 5 of 5 - Complete This One-Round Experiment

This deliberately staged experiment expects a final answer next. The following section removes that assumption and runs a real loop with repeated tool rounds and a model-request limit.


In [ ]:
final_response = client.responses.create(
    model=MODEL,
    input=conversation,
    instructions=MANUAL_INSTRUCTIONS,
    tools=MANUAL_TOOLS,
    temperature=0.1,
    max_output_tokens=600,
)
if final_response.status != "completed" or not final_response.output_text.strip():
    raise RuntimeError("Expected completed final text; inspect response status and token budget.")
remaining_calls = [item for item in final_response.output if item.type == "function_call"]
if remaining_calls:
    raise RuntimeError("This staged demo expected one tool round. Use a loop for more.")
manual_trace.append({
    "stage": "final",
    "answer": final_response.output_text,
    "response_id": final_response.id,
})

print("FINAL ANSWER:\n", final_response.output_text)
print("\nCOMPLETE MANUAL TRACE:")
print(json.dumps(manual_trace, indent=2, ensure_ascii=False))


### What We Have Built So Far

One tool round showed conversation state, schemas, argument validation, execution, `call_id`, and observations. It was intentionally not a reusable loop yet.

Next we reuse the tested implementation in `src/harness/loop.py`. Its loop is small enough to inspect in full. This is also the file you will extend for the assignment.


## Section 1B - A Real Loop: Read Data, Calculate, Answer

A dependent task prevents a convincing-looking shortcut: the model must first read exercise data, then send the retrieved expression to the calculator. `max_steps` counts **model requests**, including the final-answer request.

Expected path: `read_exercise → observation → calculate → observation → final`. We do not supply the numeric answer in the user query.


In [ ]:
import inspect
from IPython.display import Code, display
from src.config import Settings
from src.harness.loop import AgentHarness
from src.tools import Tool, ToolRegistry
from src.tools.examples import calculate

display(Code(inspect.getsource(AgentHarness._run_loop), language="python"))


In [ ]:
def read_exercise(exercise_id: str) -> dict[str, str]:
    """Read one controlled classroom exercise; unknown IDs are rejected."""
    if exercise_id != "loop-01":
        raise ValueError("Unknown exercise")
    return {"exercise_id": exercise_id, "expression": "17 * 3 + 8"}

exercise_tools = ToolRegistry([
    Tool("read_exercise", "Read an exercise's arithmetic expression before calculating.",
         {"type": "object", "properties": {"exercise_id": {"type": "string"}},
          "required": ["exercise_id"], "additionalProperties": False}, read_exercise),
    Tool("calculate", "Calculate the expression returned by read_exercise.",
         {"type": "object", "properties": {"expression": {"type": "string"}},
          "required": ["expression"], "additionalProperties": False}, calculate),
])
harness = AgentHarness(exercise_tools, Settings(provider="openai", model=MODEL,
                       openai_api_key=os.environ["OPENAI_API_KEY"]), client=client)
loop_result = harness.run(
    "Read exercise loop-01, calculate its expression using calculate, then give the result.",
    max_steps=4,
)
print("status:", loop_result.status, "| answer:", loop_result.answer)
for event in loop_result.trace:
    print(event["kind"], event["metadata"].get("tool", ""), event["content"])
called_tools = [e["metadata"]["tool"] for e in loop_result.trace if e["kind"] == "tool"]
assert loop_result.status == "completed", "Inspect the trace before continuing."
assert called_tools[:2] == ["read_exercise", "calculate"], called_tools
assert "59" in loop_result.answer


### Local Lab - Step Limit, Bad Arguments, and Incomplete Responses

The next cell injects scripted model responses. It executes the **same harness**, with no network calls. This makes failure conditions repeatable rather than hoping the model makes a mistake during class.

Tool errors become observations so the model may correct its next request. An incomplete model response stops the run with an explicit status. A limit is a stopped run, not a successful answer.


In [ ]:
from types import SimpleNamespace

def tool_response(name, arguments, call_id="test-call"):
    call = SimpleNamespace(type="function_call", name=name, arguments=arguments, call_id=call_id)
    return SimpleNamespace(id="scripted", status="completed", output=[call], output_text="")

def final_response_for_test(text="59"):
    return SimpleNamespace(id="scripted-final", status="completed", output=[], output_text=text)

class ScriptedClient:
    def __init__(self, responses):
        self.script = iter(responses)
        self.requests = []
        self.responses = SimpleNamespace(create=self.create)
    def create(self, **kwargs):
        self.requests.append(kwargs)
        return next(self.script)

limited = AgentHarness(exercise_tools, Settings(provider="mock"), client=ScriptedClient([
    tool_response("read_exercise", '{"exercise_id":"loop-01"}')
])).run("Read and calculate", max_steps=1)
assert limited.status == "max_steps"
print("STEP LIMIT:", limited.status)

script = ScriptedClient([
    tool_response("calculate", '{"expression": 59}', "bad"),
    tool_response("calculate", '{"expression": "17 * 3 + 8"}', "fixed"),
    final_response_for_test(),
])
recovered = AgentHarness(exercise_tools, Settings(provider="mock"), client=script).run("calculate", max_steps=3)
assert recovered.status == "completed"
assert any(e["kind"] == "error" for e in recovered.trace)
assert any(e["kind"] == "observation" and "ValidationError" in e["content"] for e in recovered.trace)
print("BAD ARGUMENT -> ERROR OBSERVATION -> CORRECTION:", recovered.answer)

incomplete = SimpleNamespace(id="cut-off", status="incomplete", output=[], output_text="")
stopped = AgentHarness(exercise_tools, Settings(provider="mock"), client=ScriptedClient([incomplete])).run("task")
assert stopped.status == "incomplete"
print("INCOMPLETE RESPONSE:", stopped.status)


### Check Your Understanding

- Why are two tool calls not enough for this task's model-request budget? The model still needs a final-answer turn.
- Where would an unknown tool or malformed JSON go? Through the same error-observation path.
- Does a loop trace reveal hidden reasoning? No: it records observable requests and results.

**Stop here for the core implementation checkpoint.** The SDK section now compares orchestration you have actually inspected and exercised.


---

## Section 2 - Rewrite the Same Task with the OpenAI Agents SDK

The next three code cells mirror the manual experiment: define the tool and `Agent`, call `Runner`, then inspect `RunResult`.

In Jupyter, use `await Runner.run(...)`. `Runner.run_sync(...)` should not be called from an already-running notebook event loop.


### SDK Cell 1 of 3 - Agent Definition


In [ ]:
from agents import Agent, Runner, ModelSettings, function_tool, trace
from agents.tracing import flush_traces

@function_tool
def sdk_search_class_notes(query: str) -> list[dict[str, str]]:
    """Search local class notes about agent loops and harnesses."""
    return search_class_notes(query)

notes_agent = Agent(
    name="Class Notes Agent",
    instructions=(
        "Use sdk_search_class_notes before answering questions about the notes. "
        "Answer concisely from the returned evidence."
    ),
    model=MODEL,
    model_settings=ModelSettings(max_tokens=600),
    tools=[sdk_search_class_notes],
)

print(notes_agent.name)
print([tool.name for tool in notes_agent.tools])


### SDK Cell 2 of 3 - Runner Call

`Runner` now owns model turns, tool execution, observation insertion, and the stop condition. The explicit `trace(...)` gives this run a recognizable dashboard name.


In [ ]:
with trace(
    "Class 5 - SDK Agent Loop",
    metadata={"lesson": "05", "comparison": "manual-vs-sdk"},
) as sdk_trace:
    sdk_result = await Runner.run(notes_agent, QUERY, max_turns=4)

flush_traces()
print(sdk_result.final_output)


### SDK Cell 3 of 3 - Inspect `RunResult`

For application output, `final_output` is usually enough. For debugging, inspect `new_items`, `raw_responses`, `last_agent`, and usage.


In [ ]:
sdk_summary = {
    "final_output": sdk_result.final_output,
    "last_agent": sdk_result.last_agent.name,
    "new_item_types": [type(item).__name__ for item in sdk_result.new_items],
    "raw_response_count": len(sdk_result.raw_responses),
    "usage": str(sdk_result.context_wrapper.usage),
    "trace_id": sdk_trace.trace_id,
}
print(json.dumps(sdk_summary, indent=2, ensure_ascii=False))


### Manual Harness vs Agents SDK

| Responsibility | Hand-written Responses loop | Agents SDK |
| --- | --- | --- |
| Define behavior | `instructions` | `Agent.instructions` |
| Describe a tool | Function JSON schema | `@function_tool` |
| Detect tool calls | Filter `response.output` | `Runner` |
| Validate arguments | Explicit Pydantic call | Generated schema + SDK argument validation |
| Execute Python | Registry dispatch | `Runner` |
| Return observation | Build `function_call_output` | `Runner` |
| Stop the loop | Repeated requests + status checks + max_steps | Final output + `max_turns` |
| Record execution | Build `manual_trace` | Built-in traces + `RunResult` |

The SDK removes orchestration boilerplate, but the application still owns tool design, permissions, business rules, sensitive-data policy, and evaluation.


---

## Section 3 - Open the Built-in Trace Viewer

Agents SDK tracing is enabled by default for normal OpenAI-backed runs. The named `trace(...)` above groups model generations and function execution into one workflow.

### [Open the OpenAI Traces dashboard](https://platform.openai.com/traces)

Search for **Class 5 - SDK Agent Loop**, then compare:

```text
manual_trace JSON                 OpenAI trace viewer
-----------------                 -------------------
task                              workflow / agent span
decision + action                 model generation + function call
observation                       function output
final                             final model generation
```

The dashboard can include prompts, model outputs, and tool data. Do not put secrets or unnecessary personal data into trace metadata or tool output.


In [ ]:
from html import escape
from IPython.display import HTML, Markdown, display

display(Markdown(f"**Trace ID:** `{sdk_trace.trace_id}`"))
display(Markdown("[Open the OpenAI Traces dashboard](https://platform.openai.com/traces)"))
rows = "".join(
    f"<tr><td>{i}</td><td>{escape(type(item).__name__)}</td></tr>"
    for i, item in enumerate(sdk_result.new_items, 1)
)
display(HTML("<table><tr><th>Order</th><th>Observed SDK item</th></tr>" + rows + "</table>"))
print("Model requests:", len(sdk_result.raw_responses))
print("Local RunResult is available even when trace export is disabled.")
print("A trace ID/flush call does not prove dashboard ingestion; verify it in your project.")


## Failure Boundaries to Discuss

We already exercised bad arguments, correction, a step limit, and incomplete output. Next consider tool timeouts, retries, permissions, trace export, and usage budgets. The small teaching harness is not a production sandbox. In particular, the calculator accepts a restricted syntax but does not impose CPU/resource limits.


## Checkpoint - Core Path Complete

You inspected one tool round, ran a two-tool dependent loop, observed bounded termination and error recovery, and compared the same notes task with Agent + Runner.

## Assignment Bridge

| Classroom experiment | Assignment file |
| --- | --- |
| Repeated requests / limits / errors | `src/harness/loop.py` |
| Validated local functions | `src/tools/base.py`, `src/tools/examples.py` |
| Observable steps and export | `src/harness/step.py`, `src/harness/trace.py` |

Extend the loop with one new read-only tool. Add a scripted test that fails before your change and passes after it. Use no paid API calls in tests. See `assignments/week-3-agent-harness.md` for required tasks.


## Congratulations - Fifth Class Complete

You implemented the mechanics behind an agent runtime and then replaced that orchestration with an SDK harness.

```text
model chooses -> harness validates and executes -> observation updates state -> loop continues
```

提示：SDK 让 loop 变短，但不会替你决定工具权限、停止条件、错误策略和评估标准。
